# 01. DiT: patchifier, time, attention, modulation и output

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Соберём вычисление из простых операторов, проверим ручные формулы и исследуем первые градиенты adaLN-Zero.

[Маршрут](../README.md) · [Точные задания MIT](../MIT_BRIDGE.md) · [Методы и source map](../SOURCE_MAP.md) · [Измерения](../reports/VALIDATION.md).

Каждый опыт: **теория → формула → формы/единицы → прогноз → код → измерение → ограниченный вывод**. CPU по умолчанию; без pretrained weights, сети и ComfyUI jobs.


**Связь с MIT:** MIT Lecture 4, notes §6.1 pp.41–46; Lab 3 Q3.1 (cells 38–39), Q3.2 (40–42), Q3.3 (43–44), Q3.4 (45–46), Q3.5 (47–48). Здесь Fourier time, patchifier, attention, modulation и output head.

[MIT course](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf).


In [ ]:
from pathlib import Path
import sys,math,json,copy
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
COURSE=next((p/'dit' for p in [Path.cwd(),*Path.cwd().parents] if (p/'dit/labkit/core.py').is_file()),None)
assert COURSE is not None,'Clone the study repository first; see dit/README.md (Colab setup).'
sys.path.insert(0,str(COURSE))
from labkit.core import (native,tiny_model,tiny_inputs,activate_gates,fresh_forward,
  load_k5_block,manual_single,manual_fused,CountingLinearField,tiny_bundle,
  clone_bundle,scalar_bundle,vector_from_bundle,dummy_text,show_source,
  check_sources,save_metrics,figure_path)
torch.set_num_threads(2);torch.manual_seed(42)
LAB='01_dit_primitives_and_tensor_geometry';metrics={}
print('Pinned source files checked:',check_sources())


## 1. Patchification — перестановка индексов плюс проекция

Для latent X∈R^(B×T×H×W×C) и patch p=(p_t,p_h,p_w):

$$
N_v=\frac{T}{p_t}\frac{H}{p_h}\frac{W}{p_w},\quad
P_{in}=p_t p_h p_w C,\qquad E=\mathrm{patchify}(X)W_e+b_e.
$$

После reshape/permute каждый patch — вектор, последняя ось содержит p_t,p_h,p_w,C в этом порядке. Формы: B,N_v,P_in → B,N_v,D_v. Изменение patch size меняет token budget, attention N_v² и детали, но не даёт автоматически такого же изменения всей peak memory.


In [ ]:
x=torch.arange(1*2*4*4*3,dtype=torch.float32).reshape(1,2,4,4,3)
def patchify(x):
    B,T,H,W,C=x.shape
    return x.view(B,T,1,H//2,2,W//2,2,C).permute(0,1,3,5,2,4,6,7).flatten(4,7)
def unpatchify(p,C):
    B,T,H,W,_=p.shape
    return p.view(B,T,H,W,1,2,2,C).permute(0,1,4,2,5,3,6,7).reshape(B,T,H*2,W*2,C)
p=patchify(x);assert torch.equal(unpatchify(p,3),x)
embed=native().VisualEmbeddings(3,24,(1,2,2))
assert torch.equal(embed(x),embed.in_layer(p))
metrics['patch_shapes']={'input':list(x.shape),'patches':list(p.shape),'embedding':list(embed(x).shape)}
print(metrics['patch_shapes'])


## 2. Time embedding и adaLN

Fourier features переводят scalar network time в вектор:

$$
\omega_j=\exp[-j\log(10000)/m],\qquad
f(t)=[\cos(t\omega_j),\sin(t\omega_j)]_j.
$$

TimeEmbeddings применяет Linear→SiLU→Linear; затем к time embedding добавляется projected pooled text. Modulation: SiLU→Linear выдаёт shift b, scale s и gate g. Для residual branch:

$$
\mathrm{M}(x;c)=(1+s(c))\odot\mathrm{LN}(x)+b(c),\qquad
y=x+g(c)\odot F(\mathrm{M}(x;c)).
$$

По chain rule:

$$
\frac{\partial y}{\partial\theta_F}=g\odot\frac{\partial F}{\partial\theta_F},\qquad
\frac{\partial y}{\partial g}=F(\mathrm{M}(x;c)).
$$

При g=0 первый gradient равен нулю, второй в общем случае ненулевой. У source Modulation выходная linear zero initialized. Значит сначала b=s=g=0, residual branch выключен. Ветвь может обучаться: gradient gate ненулевой, хотя gradient внутренних F weights на первом backward равен нулю. Не путайте identity отдельного transformer block с zero output всей сети.


In [ ]:
block=native().TransformerDecoderBlock(24,16,48,8,engine='sdpa')
x=torch.randn(1,6,24);text=torch.randn(1,4,24);c=torch.randn(1,16)
rope=native('core.components.rope').RoPE1D(8)(torch.arange(6))
y=block(x,text,c,rope);assert torch.equal(y,x)
loss=y.square().mean();loss.backward()
first_inner=float(block.self_attention.to_query.weight.grad.abs().max())
first_gate=float(block.visual_modulation.out_layer.weight.grad.abs().max())
assert first_inner==0 and first_gate>0
metrics['adaLN_zero']={'identity_error':float((y-x).abs().max()),'inner_weight_gradient':first_inner,'modulation_gradient':first_gate}
print(metrics['adaLN_zero'])


## 2b. LayerNorm, RMSNorm и FFN без скрытых формул

LayerNorm вычисляет статистики по последней feature axis D отдельно для каждого token, а не по sequence length:

$$
\mu=\frac1D\sum_jx_j,\quad v=\frac1D\sum_j(x_j-\mu)^2,\qquad
\mathrm{LN}(x)_j=\frac{x_j-\mu}{\sqrt{v+\epsilon}}.
$$

У block pre-norm elementwise_affine=False: learned shift/scale поступают из modulation. У text projection LayerNorm affine=True. RMSNorm для Q/K не вычитает mean:

$$
\mathrm{RMSNorm}(x)_j=w_j\frac{x_j}{\sqrt{D^{-1}\sum_i x_i^2+\epsilon}}.
$$

Поскольку E[x²]=Var(x)+E[x]², RMSNorm и LN совпадают только при дополнительных условиях; сдвиг x константой не нейтрален для RMSNorm.

FFN source использует bias-free Linears и exact GELU; dropout=0:

$$
\mathrm{GELU}(x)=\frac{x}{2}\left[1+\mathrm{erf}\left(\frac{x}{\sqrt2}\right)\right],\qquad
\mathrm{FFN}(x)=W_2\,\mathrm{GELU}(W_1x).
$$

Для row-vector code запись эквивалентна xW₁ᵀ и затем W₂ᵀ. FFN действует независимо по tokens, attention связывает tokens. Проверим оба norm и GELU по формулам.


In [ ]:
x=torch.randn(2,5,8);ln=nn.LayerNorm(8,elementwise_affine=False);rms=nn.RMSNorm(8)
mean=x.mean(-1,keepdim=True);var=(x-mean).square().mean(-1,keepdim=True)
manual_ln=(x-mean)/torch.sqrt(var+ln.eps)
eps=rms.eps if rms.eps is not None else torch.finfo(x.dtype).eps
manual_rms=x/torch.sqrt(x.square().mean(-1,keepdim=True)+eps)*rms.weight
manual_gelu=.5*x*(1+torch.erf(x/math.sqrt(2)))
assert torch.allclose(ln(x),manual_ln,atol=1e-6)
assert torch.allclose(rms(x),manual_rms,atol=1e-6)
assert torch.allclose(F.gelu(x),manual_gelu,atol=1e-6)
metrics['normalization_formula_errors']={'LayerNorm':float((ln(x)-manual_ln).abs().max()),'RMSNorm':float((rms(x)-manual_rms).abs().max()),'GELU':float((F.gelu(x)-manual_gelu).abs().max())}
# Explicit modulation broadcast over tokens for B=2.
scale=torch.randn(2,8);shift=torch.randn(2,8)
actual=native('core.tensors').apply_scale_shift_norm(ln,x,scale,shift)
assert torch.allclose(actual,ln(x)*(1+scale[:,None])+shift[:,None])


## 3. Attention от скалярных произведений к tensor axes

При h heads, d=D/h получаем Q,K,V размера B,h,N,d. Scores делятся на sqrt(d), чтобы при независимых unit-variance coordinates дисперсия суммы не росла как d:

$$
S_{ij}=\frac{q_i^\top k_j}{\sqrt d}+M_{ij},\quad
A_{ij}=\frac{\exp S_{ij}}{\sum_{l}\exp S_{il}},\quad Y_i=\sum_j A_{ij}v_j.
$$

В K6 Q/K дополнительно имеют RMSNorm по head dimension и RoPE. Orthogonal rotation сохраняет нормы, но dot product двух разных positions меняет через relative rotation. Это не добавление позиции к latent value и не attention mask.

Полный forward ниже: Linears → reshape → RMSNorm → rotate → scaled softmax → weighted V → output Linear. Сравниваем ручную формулу с оригинальной attention, использующей локальный dense-SDPA backend.


In [ ]:
attn=native().MultiheadSelfAttentionDec(24,8,engine='sdpa');x=torch.randn(1,6,24)
q=attn.query_norm(attn.to_query(x).reshape(1,6,3,8))
k=attn.key_norm(attn.to_key(x).reshape(1,6,3,8));v=attn.to_value(x).reshape(1,6,3,8)
rot=native('core.tensors').apply_rotary
qr,kr=rot(q,rope),rot(k,rope)
A=torch.softmax(qr.transpose(1,2)@kr.transpose(1,2).transpose(-1,-2)/math.sqrt(8),dim=-1)
manual=attn.out_layer((A@v.transpose(1,2)).transpose(1,2).flatten(-2,-1))
actual=attn(x,rope)
assert torch.allclose(manual,actual,atol=2e-6)
assert torch.allclose(qr.norm(dim=-1),q.norm(dim=-1),atol=2e-6)
metrics['attention_manual_error']=float((manual-actual).abs().max())
fig,ax=plt.subplots(figsize=(4.5,3.5));im=ax.imshow(A[0,0].detach());fig.colorbar(im,ax=ax);ax.set(xlabel='Key token',ylabel='Query token',title='One attention head; row sums = 1')
fig.tight_layout();fig.savefig(figure_path(LAB,'attention'));plt.show()


## 4. OutLayer — learned prediction, не обратная embedding matrix

Hidden tokens проходят conditioned normalization и Linear в p_t p_h p_w C_out значений. Source video output layout раскладывает последнее измерение как **C_out,p_t,p_h,p_w**, затем переставляет в B,T,H,W,C_out. Это отличается от порядка входного patch vector. Learned output projection учитывает своё соглашение; нельзя объявить OutLayer inverse VisualEmbeddings с identity weights.

Audio head выдаёт B,A,C_a и содержит **два** применения norm. Второй norm есть в pinned training-compatible code; при переносе его нельзя удалять как «лишний» без parity test.

Выполним whole random tiny K6 forward: обычная голова сохраняет physical dimensions, grid-head позже добавит отдельную ось G. Это проверка shapes/function, не generation quality.


In [ ]:
model=tiny_model().eval();v,a,kw=tiny_inputs(model)
with torch.no_grad():pv,pa=fresh_forward(model,kw)
assert pv.shape==v.shape and pa.shape==a.shape
metrics['native_tiny_shapes']={'video':list(pv.shape),'audio':list(pa.shape),'parameters':sum(p.numel() for p in model.parameters())}
show_source('core/components/dit.py','OutLayerAudio')
print(metrics['native_tiny_shapes'])


## Измерение → вывод

Patch indices, attention formula, rotation norms и zero-init gradient routing проверены. Full tiny model сохранил размеры; никакого видео/аудио качества случайной сети эти тесты не оценивают.


## Самостоятельные опыты

1. Закончите MIT Q3.1–Q3.5 и докажите patch roundtrip для p_t=2.
2. После одного optimizer step снова измерьте gradient attention weights.
3. Увеличьте N в 2 раза и посчитайте score elements отдельно от total memory.
4. Выведите relative-position property Rᵀ(i)R(j)=R(j−i).

Перед изменением запишите прогноз. Сохраняйте baseline, меняйте один фактор, отмечайте seed и tensor layout. Нулевая ошибка на oracle не равна pretrained quality.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Первичные источники

[MIT course / notes](https://diffusion.csail.mit.edu/2026/index.html) · [DiT](https://arxiv.org/abs/2212.09748) · [Flow Matching](https://arxiv.org/abs/2210.02747) · [K5](https://arxiv.org/abs/2511.14993) · [K6 §3.2, §4.5](https://arxiv.org/html/2610.05608v1) · [π-Flow §3.1, §4 / Alg.1](https://arxiv.org/html/2510.14974v1) · [Source hashes / provenance](../sources.json).

Формулы ниже/выше выводятся для явно заданных соглашений. Тела source methods сохраняются; adapters и doubles подписаны. Малые experiments не воспроизводят промышленное обучение.
